# QRQ-FL — full implementation (Colab)

This notebook runs the **same stack as `main.py`** end-to-end:

* **Real vision data**: CIFAR-10 or CIFAR-100 (torchvision download).
* **Non-IID split**: Dirichlet label skew (`data/partition.py`).
* **Flower simulation**: `fl.simulation.start_simulation`.
* **Clients**: `QRQClient` — local SGD + simulated PQ delay from measured/stub timings.
* **Server**: `QRQFLStrategy` — HMM dropout beliefs (`hmm/reliability.py`), DPP client + PQ selection (`rl/dpp.py`), virtual queues, FedAvg aggregation.

You get a **FedAvg** baseline (random client sampling) and **QRQ-FL** (DPP + HMM + PQ set), with **centralized test accuracy each round**, metric CSV, plots, and `results.json`.

**Colab:** enable GPU (Runtime → Change runtime type). The last cell downloads `figures.zip` and `results.json` when `google.colab` is available.

**Note:** `fl.simulation.start_simulation` needs **Ray**. This notebook installs `flwr[simulation]` (not plain `flwr`) so Ray is pulled in automatically.

**Note:** Flower schedules virtual clients with `num_gpus=0`; Ray workers then have **no CUDA**. Local training uses **CPU** inside those actors; **centralized evaluation** still uses the Colab GPU when available (faster eval, correct semantics).


## 1. Environment and dependencies

In [ ]:
import sys, subprocess

COLAB = "google.colab" in sys.modules
print(f"Running on: {'Google Colab' if COLAB else 'local'}")
print(f"Python:     {sys.version.split()[0]}")


def pip_install(pkg: str) -> None:
    print(f"  installing {pkg} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


def have_ray() -> bool:
    try:
        import ray  # noqa: F401
        return True
    except ImportError:
        return False


# Flower simulation backend requires Ray — use the official extra (see flwr error message).
FLWR_SIM = "flwr[simulation]>=1.8.0,<2"
try:
    import flwr  # noqa: F401
    if not have_ray():
        pip_install(FLWR_SIM)
except ImportError:
    pip_install(FLWR_SIM)

for pkg in ("matplotlib", "pandas", "tqdm"):
    try:
        if pkg == "matplotlib":
            import matplotlib  # noqa: F401
        elif pkg == "pandas":
            import pandas  # noqa: F401
        else:
            import tqdm  # noqa: F401
    except ImportError:
        pip_install(pkg)

import torch
print(f"torch:      {torch.__version__}")
import flwr as fl
print(f"flwr:       {fl.__version__}")
if not have_ray():
    raise RuntimeError("Ray is still missing after installing flwr[simulation]. Run: pip install -U 'flwr[simulation]'")
import ray
print(f"ray:        {ray.__version__}")


## 2. Imports, seed, device

In [ ]:
from __future__ import annotations

import csv
import json
import pathlib
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

import flwr as fl
from flwr.common import Parameters, parameters_to_ndarrays

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_TIER = "cpu"
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0).lower()
    if "a100" in name:
        GPU_TIER = "a100"
    elif "v100" in name:
        GPU_TIER = "v100"
    elif "l4" in name:
        GPU_TIER = "l4"
    elif "t4" in name:
        GPU_TIER = "t4"
    else:
        GPU_TIER = "gpu_other"
    print(f"Device: {DEVICE} | {torch.cuda.get_device_name(0)} | tier={GPU_TIER}")
else:
    print(f"Device: {DEVICE} (no GPU)")

# Centralized eval runs in the notebook kernel (sees GPU). Flower Ray actors use
# client_resources num_gpus=0.0 — those workers do not get a CUDA device, so
# passing DEVICE="cuda" into QRQClient causes: No CUDA GPUs are available.
CLIENT_DEVICE = "cpu" if DEVICE == "cuda" else DEVICE
print(f"Client train device (Flower simulation): {CLIENT_DEVICE}  (server eval: {DEVICE})")


## 3. Hyperparameters

Federation size defaults scale with `GPU_TIER`. Override by editing the dict literals.


In [ ]:
# --- Dataset ---
DATASET = "cifar100"  # "cifar10" | "cifar100"

PRESETS = {
    "cpu":       dict(N_CLIENTS=10, N_SELECT=5,  N_ROUNDS=8,  BATCH_SIZE=32),
    "t4":        dict(N_CLIENTS=20, N_SELECT=10, N_ROUNDS=15, BATCH_SIZE=64),
    "l4":        dict(N_CLIENTS=30, N_SELECT=12, N_ROUNDS=20, BATCH_SIZE=128),
    "v100":      dict(N_CLIENTS=40, N_SELECT=12, N_ROUNDS=25, BATCH_SIZE=128),
    "a100":      dict(N_CLIENTS=50, N_SELECT=15, N_ROUNDS=30, BATCH_SIZE=256),
    "gpu_other": dict(N_CLIENTS=20, N_SELECT=10, N_ROUNDS=15, BATCH_SIZE=64),
}
preset = PRESETS[GPU_TIER]
N_CLIENTS = preset["N_CLIENTS"]
N_SELECT = preset["N_SELECT"]
N_ROUNDS = preset["N_ROUNDS"]
BATCH_SIZE = preset["BATCH_SIZE"]

DIRICHLET_ALPHA = 0.3
LOCAL_EPOCHS = 1
LR = 0.01
MOMENTUM = 0.9

# DPP / QRQ (matches main.py defaults)
DPP_V = 2.0
PQ_SET = ("kyber512", "dilithium2", "falcon-512")
QRQ_DEADLINE_S = 2.0

# Run FedAvg baseline first (doubles wall time; set False for QRQ-FL only)
RUN_FEDAVG_BASELINE = True

print(json.dumps({
    "DATASET": DATASET,
    "N_CLIENTS": N_CLIENTS,
    "N_SELECT": N_SELECT,
    "N_ROUNDS": N_ROUNDS,
    "BATCH_SIZE": BATCH_SIZE,
    "DIRICHLET_ALPHA": DIRICHLET_ALPHA,
    "DEVICE": DEVICE,
    "CLIENT_DEVICE": CLIENT_DEVICE,
}, indent=2))


## 4. Real dataset + Dirichlet non-IID split (`data/partition.py`)

In [ ]:
"""Non-IID partitioners shared by all datasets.

  - dirichlet_partition()  : label-skew via Dirichlet(alpha)  (CIFAR-100)
  - per_writer_partition() : returns the LEAF FEMNIST per-writer split
                             (call after running LEAF preprocess.sh)
  - hospital_partition()   : MIMIC-III hospital-as-client
"""
from __future__ import annotations

import json
import pathlib
from collections import defaultdict

import numpy as np


def dirichlet_partition(labels: np.ndarray, n_clients: int, alpha: float = 0.3, seed: int = 0):
    """Assign sample indices to clients with label-Dirichlet skew.
    Returns: list[np.ndarray] of length n_clients."""
    rng = np.random.default_rng(seed)
    n_classes = int(labels.max()) + 1
    idx_by_class = [np.where(labels == c)[0] for c in range(n_classes)]
    for arr in idx_by_class:
        rng.shuffle(arr)
    splits = [[] for _ in range(n_clients)]
    for c in range(n_classes):
        proportions = rng.dirichlet([alpha] * n_clients)
        cuts = (np.cumsum(proportions) * len(idx_by_class[c])).astype(int)[:-1]
        for k, part in enumerate(np.split(idx_by_class[c], cuts)):
            splits[k].extend(part.tolist())
    return [np.array(sorted(s), dtype=np.int64) for s in splits]


def per_writer_partition(leaf_femnist_dir: str | pathlib.Path):
    """Read LEAF FEMNIST per-writer JSON files. Each file maps user_id ->
    {x: [...], y: [...]}. We just yield (user_id, X, y)."""
    base = pathlib.Path(leaf_femnist_dir)
    train = base / "data" / "train"
    if not train.exists():
        raise FileNotFoundError(
            f"FEMNIST not preprocessed yet: {train} missing. "
            "Run `cd leaf/data/femnist && ./preprocess.sh -s niid --sf 0.05 -k 0 -t sample`"
        )
    for jf in sorted(train.glob("*.json")):
        with open(jf, "r", encoding="utf-8") as f:
            blob = json.load(f)
        for uid in blob["users"]:
            ud = blob["user_data"][uid]
            yield uid, np.asarray(ud["x"], dtype=np.float32), np.asarray(ud["y"], dtype=np.int64)


def hospital_partition(features: np.ndarray, hospital_ids: np.ndarray):
    """Group MIMIC-III rows by hospital id."""
    parts = defaultdict(list)
    for i, h in enumerate(hospital_ids):
        parts[int(h)].append(i)
    return {h: np.array(idx, dtype=np.int64) for h, idx in parts.items()}


DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)

if DATASET == "cifar10":
    NUM_CLASSES = 10
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.247, 0.243, 0.261)
    DS = torchvision.datasets.CIFAR10
elif DATASET == "cifar100":
    NUM_CLASSES = 100
    mean = (0.5071, 0.4865, 0.4409)
    std = (0.2673, 0.2564, 0.2762)
    DS = torchvision.datasets.CIFAR100
else:
    raise ValueError(DATASET)

tfm = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
print(f"Downloading {DATASET} …")
train = DS(root=DATA_DIR, train=True, download=True, transform=tfm)
test = DS(root=DATA_DIR, train=False, download=True, transform=tfm)
labels = np.asarray(train.targets)
splits = dirichlet_partition(labels, N_CLIENTS, alpha=DIRICHLET_ALPHA, seed=SEED)
train_loaders = [
    DataLoader(Subset(train, idx.tolist()), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    for idx in splits
]
val_loader = DataLoader(test, batch_size=256, shuffle=False, num_workers=0)
sizes = [len(s) for s in splits]
print(f"Train {len(train):,} | Test {len(test):,} | clients min/med/max = {min(sizes)}/{int(np.median(sizes))}/{max(sizes)}")


## 5. CNN (same architecture as `main.py`)

In [ ]:
def small_cnn(num_classes: int, in_channels: int = 3) -> nn.Module:
    return nn.Sequential(
        nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
        nn.Linear(256, num_classes),
    )


IN_CHANNELS = 3
model_example = small_cnn(NUM_CLASSES, IN_CHANNELS).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model_example.parameters()):,}")
del model_example


## 6. HMM reliability layer (`hmm/reliability.py`)

In [ ]:
# --- inlined from hmm/reliability.py ---
"""Hidden-Markov client reliability layer.

Implements the four-state model {Online, Busy, Offline, Byzantine} from
Section III-C of the paper. Two operations matter:

  1. forward()   -> returns rho_i(t) = P[ Z_i(t) in {Offline, Byzantine} ]
                    given the observation history.
  2. fit()       -> Baum-Welch EM on a batch of observation sequences.

We deliberately keep the implementation in NumPy (no hmmlearn dependency
during inference) so it can be embedded inside the FL server's hot path.
hmmlearn is only used when fit_hmmlearn() is called for a more robust EM.
"""
from __future__ import annotations

import numpy as np
from dataclasses import dataclass

ONLINE, BUSY, OFFLINE, BYZ = 0, 1, 2, 3
N_STATES = 4
DROPOUT_MASK = np.array([0.0, 0.0, 1.0, 1.0])  # Offline + Byzantine count as drop


@dataclass
class HMMParams:
    """Per-client HMM: transition P (4x4), emission B (4x|O|), prior pi (4,)."""
    P: np.ndarray
    B: np.ndarray
    pi: np.ndarray

    @classmethod
    def reasonable_default(cls, n_obs: int = 3) -> "HMMParams":
        # Sticky transitions: clients tend to stay in their current state.
        P = np.array([
            [0.85, 0.08, 0.05, 0.02],
            [0.20, 0.70, 0.08, 0.02],
            [0.15, 0.05, 0.78, 0.02],
            [0.02, 0.03, 0.05, 0.90],
        ])
        # Emission: 0=ok-update, 1=missing, 2=deviation-flag.
        B = np.array([
            [0.90, 0.05, 0.05],
            [0.50, 0.45, 0.05],
            [0.00, 1.00, 0.00],
            [0.20, 0.10, 0.70],
        ])
        if n_obs != 3:
            B = np.ones((N_STATES, n_obs)) / n_obs
        pi = np.array([0.7, 0.2, 0.08, 0.02])
        return cls(P=P, B=B, pi=pi)


def forward(obs: np.ndarray, hp: HMMParams) -> np.ndarray:
    """Forward algorithm: returns alpha[t, k] = P(O_{1:t}, Z_t=k)."""
    T = len(obs)
    alpha = np.zeros((T, N_STATES))
    alpha[0] = hp.pi * hp.B[:, obs[0]]
    s = alpha[0].sum()
    if s > 0:
        alpha[0] /= s  # rescale to prevent underflow; this turns alpha into filtered posterior.
    for t in range(1, T):
        alpha[t] = (alpha[t - 1] @ hp.P) * hp.B[:, obs[t]]
        s = alpha[t].sum()
        if s > 0:
            alpha[t] /= s
    return alpha


def dropout_probability(obs: np.ndarray, hp: HMMParams) -> float:
    """rho_i(t) used by the RL scheduler in eq. (Expected Participation)."""
    alpha = forward(obs, hp)
    return float(alpha[-1] @ DROPOUT_MASK)


def expected_participation(obs_per_client: list[np.ndarray], hp: HMMParams) -> float:
    """bar n(t) / N from the paper."""
    return 1.0 - np.mean([dropout_probability(o, hp) for o in obs_per_client])


def fit_baum_welch(
    sequences: list[np.ndarray],
    n_obs: int,
    n_iter: int = 50,
    tol: float = 1e-4,
    seed: int = 0,
) -> HMMParams:
    """Lightweight EM for the four-state HMM. sequences is a list of
    observation arrays (one per client), each containing integers in
    [0, n_obs)."""
    rng = np.random.default_rng(seed)
    P = rng.dirichlet(np.ones(N_STATES), size=N_STATES)
    B = rng.dirichlet(np.ones(n_obs), size=N_STATES)
    pi = rng.dirichlet(np.ones(N_STATES))

    prev_ll = -np.inf
    for it in range(n_iter):
        # Sufficient statistics
        gamma_sum = np.zeros(N_STATES)
        xi_sum = np.zeros((N_STATES, N_STATES))
        emit_sum = np.zeros((N_STATES, n_obs))
        pi_sum = np.zeros(N_STATES)
        ll = 0.0

        for obs in sequences:
            T = len(obs)
            if T < 2:
                continue
            alpha = np.zeros((T, N_STATES))
            beta = np.zeros((T, N_STATES))
            scale = np.zeros(T)
            alpha[0] = pi * B[:, obs[0]]
            scale[0] = alpha[0].sum() or 1.0
            alpha[0] /= scale[0]
            for t in range(1, T):
                alpha[t] = (alpha[t - 1] @ P) * B[:, obs[t]]
                scale[t] = alpha[t].sum() or 1.0
                alpha[t] /= scale[t]
            beta[T - 1] = 1.0
            for t in range(T - 2, -1, -1):
                beta[t] = P @ (B[:, obs[t + 1]] * beta[t + 1])
                beta[t] /= scale[t + 1]

            ll += np.log(scale + 1e-300).sum()
            gamma = alpha * beta
            gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
            for t in range(T - 1):
                xi = (alpha[t][:, None] * P) * (B[:, obs[t + 1]] * beta[t + 1])[None, :]
                xi /= xi.sum() + 1e-300
                xi_sum += xi
            gamma_sum += gamma[:-1].sum(axis=0)
            for t, o in enumerate(obs):
                emit_sum[:, o] += gamma[t]
            pi_sum += gamma[0]

        # M-step
        P = xi_sum / (gamma_sum[:, None] + 1e-300)
        B = emit_sum / (emit_sum.sum(axis=1, keepdims=True) + 1e-300)
        pi = pi_sum / (pi_sum.sum() + 1e-300)

        if abs(ll - prev_ll) < tol:
            break
        prev_ll = ll

    return HMMParams(P=P, B=B, pi=pi)


## 7. PQ timing (`pqc/timing.py`)

In [ ]:
# --- inlined from pqc/timing.py ---
"""Empirical timing for the post-quantum primitives in Table I.

Tries to load liboqs-python; if unavailable, falls back to a deterministic
stub model whose mean cycle counts come from published OQS benchmarks.
The stub keeps the rest of the pipeline reproducible without native deps.
"""
from __future__ import annotations

import statistics
import time
from dataclasses import dataclass

# CPU cycles -> seconds at this clock rate (used in eq. (pq_service)).
DEFAULT_CPU_GHZ = 3.2

# Order-of-magnitude cycle counts from the OQS reference implementations
# (used only as a stub when liboqs is not available; do not rely on for
# publishable numbers, run the real benchmark on the target hardware).
STUB_CYCLES = {
    "kyber512":              30_000,
    "ntru-hps-2048-509":     45_000,
    "frodokem-640-aes":     900_000,
    "dilithium2":           250_000,
    "falcon-512":         3_500_000,
}


@dataclass
class PQTiming:
    algo: str
    op: str        # "keypair" | "encaps" | "decaps" | "sign" | "verify"
    samples: list[float]

    @property
    def mean(self) -> float:
        return statistics.mean(self.samples) if self.samples else 0.0

    @property
    def p95(self) -> float:
        return float(sorted(self.samples)[int(0.95 * len(self.samples))]) if self.samples else 0.0


def _measure_liboqs(algo: str, n: int = 100) -> PQTiming | None:
    """Real measurement via liboqs-python."""
    try:
        import oqs  # type: ignore
    except Exception:
        return None

    if algo in oqs.get_enabled_kem_mechanisms():
        kem = oqs.KeyEncapsulation(algo)
        pk = kem.generate_keypair()
        samples = []
        for _ in range(n):
            t0 = time.perf_counter()
            ct, ss = kem.encap_secret(pk)
            samples.append(time.perf_counter() - t0)
        kem.free()
        return PQTiming(algo=algo, op="encaps", samples=samples)

    if algo in oqs.get_enabled_sig_mechanisms():
        sig = oqs.Signature(algo)
        pk = sig.generate_keypair()
        msg = b"qrqfl-bench"
        samples = []
        for _ in range(n):
            t0 = time.perf_counter()
            sig.sign(msg)
            samples.append(time.perf_counter() - t0)
        sig.free()
        return PQTiming(algo=algo, op="sign", samples=samples)

    return None


def _measure_stub(algo: str, n: int = 100, cpu_ghz: float = DEFAULT_CPU_GHZ) -> PQTiming:
    cycles = STUB_CYCLES.get(algo.lower())
    if cycles is None:
        raise ValueError(f"Unknown algo for stub: {algo}")
    mean = cycles / (cpu_ghz * 1e9)
    # Add ~5% jitter so the queue model's S^2 moment is non-degenerate.
    samples = [mean * (1.0 + 0.05 * ((i % 7) - 3) / 3) for i in range(n)]
    return PQTiming(algo=algo, op="stub", samples=samples)


def measure(algo: str, n: int = 100) -> PQTiming:
    real = _measure_liboqs(algo, n)
    if real is not None and real.samples:
        return real
    return _measure_stub(algo, n)


## 8. DPP scheduler (`rl/dpp.py`)

In [ ]:
# --- inlined from rl/dpp.py ---
"""Drift-Plus-Penalty per-slot action selector (eq. (dpp_action) of the paper).

Solves
    a*_t in argmax_a  V * r(s,a)  -  sum_i Q_i(t) g_i(s,a)
over a finite candidate set generated from the available clients and the
PQ-scheme library. For very large action spaces, swap this for the PPO
agent in stable_baselines3 (see fl/strategy.py for the hook).
"""
from __future__ import annotations

import itertools
import math
from dataclasses import dataclass

import numpy as np


@dataclass
class State:
    rho: np.ndarray              # per-client dropout posterior, shape (N,)
    snr: np.ndarray              # per-client SNR proxy,         shape (N,)
    queue_lengths: np.ndarray    # virtual queue backlogs Q_i,   shape (M,)
    pq_service_time: dict[str, float]   # {algo: E[S_pq]}


@dataclass
class Action:
    selected: np.ndarray         # binary, shape (N,)
    pq_scheme: str
    bandwidth: np.ndarray        # per-selected-client share, shape (N,)


def reward(state: State, action: Action, *, w_acc=1.0, w_lat=0.5, w_eng=0.1) -> float:
    """Single-step reward used by both the DPP optimiser and PPO."""
    sel = action.selected.astype(bool)
    if not sel.any():
        return -1.0  # no participation -> heavy penalty
    expected_acc_gain = float(w_acc * (1.0 - state.rho[sel]).mean())
    expected_latency  = state.pq_service_time[action.pq_scheme]
    expected_energy   = float(sel.mean())
    return expected_acc_gain - w_lat * expected_latency - w_eng * expected_energy


def constraints(state: State, action: Action) -> np.ndarray:
    """g_i(s,a) for each virtual queue. Negative is good (strictly feasible)."""
    sel = action.selected.astype(bool)
    util = sel.mean()
    pq_cost = state.pq_service_time[action.pq_scheme] - 0.10  # SLA: 100 ms
    if len(state.rho) == len(sel):
        drop_cost = float(state.rho[sel].mean()) - 0.30 if sel.any() else 0.0
    else:
        drop_cost = float(np.mean(state.rho)) - 0.30 if sel.any() else 0.0
    return np.array([util - 0.6, pq_cost, drop_cost])


def candidate_actions(state: State, k: int, pq_set: list[str], samples: int = 256, seed: int = 0):
    """Generate `samples` random feasible actions: pick top-2k by reliability,
    sample k of them, and try every PQ scheme."""
    rng = np.random.default_rng(seed)
    N = len(state.rho)
    pool = np.argsort(state.rho)[: min(2 * k, N)]
    for _ in range(samples):
        chosen = rng.choice(pool, size=min(k, len(pool)), replace=False)
        sel = np.zeros(N, dtype=int)
        sel[chosen] = 1
        bw = np.zeros(N)
        bw[chosen] = 1.0 / max(len(chosen), 1)
        for pq in pq_set:
            yield Action(selected=sel, pq_scheme=pq, bandwidth=bw)


def dpp_select(state: State, V: float, k: int, pq_set: list[str], samples: int = 256) -> Action:
    """Per-slot DPP optimiser."""
    best, best_score = None, -math.inf
    for a in candidate_actions(state, k, pq_set, samples):
        r = reward(state, a)
        g = constraints(state, a)
        score = V * r - float(state.queue_lengths @ np.maximum(g, 0))
        if score > best_score:
            best_score, best = score, a
    assert best is not None
    return best


def update_virtual_queues(Q: np.ndarray, g: np.ndarray) -> np.ndarray:
    """Eq. (virtual_queue): Q_i(t+1) = max(Q_i(t) + g_i, 0)."""
    return np.maximum(Q + g, 0.0)


## 9. Flower client (`fl/client.py`)

In [ ]:
# --- inlined from fl/client.py ---
"""Minimal Flower NumPyClient for CIFAR-100 / FEMNIST / IoV.

Each client is given (model_factory, train_loader, val_loader, device).
The PQ encryption step is simulated by adding a configurable sleep that
follows the empirical S_pq distribution; this lets the queueing layer
observe realistic per-round service times without depending on a working
liboqs install.
"""
from __future__ import annotations

import time
from typing import Callable

import numpy as np
import torch
import flwr as fl


class QRQClient(fl.client.NumPyClient):
    def __init__(
        self,
        client_id: str,
        model_factory: Callable[[], torch.nn.Module],
        train_loader,
        val_loader,
        device: str = "cpu",
        local_epochs: int = 1,
        pq_overhead_sampler: Callable[[str], float] | None = None,
    ):
        self.cid = client_id
        self.model_factory = model_factory
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.local_epochs = local_epochs
        self.pq_overhead_sampler = pq_overhead_sampler or (lambda algo: 0.0)
        self.model = self.model_factory().to(self.device)

    # --- Flower API ----------------------------------------------------
    def get_parameters(self, config):
        return [v.detach().cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        sd = self.model.state_dict()
        for k, v in zip(sd.keys(), parameters):
            sd[k] = torch.tensor(v)
        self.model.load_state_dict(sd, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        # Snapshot the global parameters for the optional FedProx proximal term.
        global_snapshot = [
            torch.tensor(p, device=self.device).detach() for p in parameters
        ]
        mu = float(config.get("proximal_mu", 0.0) or 0.0)
        opt = torch.optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)
        loss_fn = torch.nn.CrossEntropyLoss()
        self.model.train()
        for _ in range(self.local_epochs):
            for xb, yb in self.train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                opt.zero_grad()
                loss = loss_fn(self.model(xb), yb)
                if mu > 0.0:
                    prox = 0.0
                    for p, g in zip(self.model.parameters(), global_snapshot):
                        prox = prox + ((p - g) ** 2).sum()
                    loss = loss + 0.5 * mu * prox
                loss.backward()
                opt.step()
        # Simulate PQ encryption overhead so the server-side queue sees it.
        time.sleep(self.pq_overhead_sampler(config.get("pq_scheme", "kyber512")))
        n = sum(len(b[0]) for b in self.train_loader)
        return self.get_parameters({}), n, {"client_id": self.cid}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        correct = total = 0
        loss_total = 0.0
        loss_fn = torch.nn.CrossEntropyLoss(reduction="sum")
        with torch.no_grad():
            for xb, yb in self.val_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                logits = self.model(xb)
                loss_total += loss_fn(logits, yb).item()
                correct += (logits.argmax(1) == yb).sum().item()
                total += len(yb)
        return loss_total / max(total, 1), total, {"accuracy": correct / max(total, 1)}


## 10. QRQ-FL strategy (`fl/strategy.py`)

In [ ]:
# --- inlined from fl/strategy.py (package imports omitted; see cells above) ---
measure_pq = measure  # alias used inside QRQFLStrategy
"""Custom Flower strategy that wires the four QRQ-FL layers together.

The base FedAvg algorithm is unchanged; what we add are the three hooks
described in Algorithm 1 of the paper:

  configure_fit()   -> uses HMM rho_i(t) + DPP optimiser to pick clients
  aggregate_fit()   -> applies homomorphic mask cancellation (DTAHE stub)
                       and updates the virtual queues Q_i(t)
  evaluate()        -> records latency / hat-gamma / PQ overhead per round
"""
from __future__ import annotations

import time
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import flwr as fl
from flwr.common import (
    EvaluateIns, EvaluateRes, FitIns, FitRes, Parameters, Scalar,
    parameters_to_ndarrays, ndarrays_to_parameters,
)
from flwr.server.client_manager import ClientManager
from flwr.server.client_proxy import ClientProxy



@dataclass
class QRQConfig:
    n_select: int = 10
    V: float = 2.0
    pq_set: tuple[str, ...] = ("kyber512", "dilithium2", "falcon-512")
    deadline_s: float = 2.0
    # --- ablation / variant flags ---
    # use_hmm=False  -> rho_i is replaced by a uniform value (no reliability bias)
    # use_dpp=False  -> selection ignores virtual queues and reward; picks top-k
    #                   by reliability (or random when use_hmm=False)
    # fixed_pq       -> when set, the chosen scheme is forced every round; this
    #                   disables PQ-arm of DPP but leaves client selection intact
    # proximal_mu    -> if > 0, broadcast a FedProx mu so clients add a proximal
    #                   regulariser to their local SGD loss
    use_hmm: bool = True
    use_dpp: bool = True
    fixed_pq: str | None = None
    proximal_mu: float = 0.0


class QRQFLStrategy(fl.server.strategy.FedAvg):
    """FedAvg + HMM + queueing + DPP + (optional) PQ stub.

    Ablation modes are selected via the ``QRQConfig`` flags; setting them all
    off reduces the strategy to plain FedAvg with random client sampling.
    """

    def __init__(self, *args, qrq: QRQConfig | None = None, hmm: HMMParams | None = None, **kw):
        super().__init__(*args, **kw)
        self.cfg = qrq or QRQConfig()
        self.hmm = hmm or HMMParams.reasonable_default()
        self._obs_history: dict[str, list[int]] = defaultdict(list)
        self._Q = np.zeros(3)
        self.metrics: list[dict] = []
        self._pq_cache: dict[str, float] = {
            algo: measure_pq(algo, n=20).mean for algo in self.cfg.pq_set
        }
        self._rng = np.random.default_rng(0)

    # --- selection ---------------------------------------------------
    def _select_clients(self, all_clients, rho: np.ndarray):
        """Pick the round's selected clients honouring use_hmm / use_dpp."""
        n = len(all_clients)
        k = min(self.cfg.n_select, n)
        if self.cfg.use_dpp:
            snr = np.full(n, 20.0)
            state = State(rho=rho, snr=snr, queue_lengths=self._Q,
                          pq_service_time=self._pq_cache)
            action = dpp_select(state, V=self.cfg.V, k=k,
                                pq_set=list(self.cfg.pq_set))
            chosen_idx = [i for i, x in enumerate(action.selected) if x]
            scheme = action.pq_scheme
        else:
            if self.cfg.use_hmm:
                chosen_idx = list(np.argsort(rho)[:k])
            else:
                chosen_idx = list(self._rng.choice(n, size=k, replace=False))
            scheme = self.cfg.fixed_pq or next(iter(self.cfg.pq_set))
        if self.cfg.fixed_pq is not None:
            scheme = self.cfg.fixed_pq
        return chosen_idx, scheme

    def configure_fit(self, server_round, parameters, client_manager: ClientManager):
        all_clients = list(client_manager.all().values())
        if not all_clients:
            return []
        cids = [c.cid for c in all_clients]

        if self.cfg.use_hmm:
            rho = np.array([
                dropout_probability(np.asarray(self._obs_history[cid] or [0]), self.hmm)
                for cid in cids
            ])
        else:
            rho = np.full(len(cids), 0.5)

        chosen_idx, scheme = self._select_clients(all_clients, rho)
        chosen = [all_clients[i] for i in chosen_idx]
        cfg = {
            "round": server_round,
            "pq_scheme": scheme,
            "deadline_s": self.cfg.deadline_s,
            "proximal_mu": float(self.cfg.proximal_mu),
        }
        return [(c, FitIns(parameters, cfg)) for c in chosen]

    # --- aggregation -------------------------------------------------
    def aggregate_fit(self, server_round, results, failures):
        # Update HMM observation history (0=ok, 1=missing).
        observed_ok = {res.metrics.get("client_id", c.cid) for c, res in results}
        for c, _ in results:
            self._obs_history[c.cid].append(0)
        for cf in failures:
            cid = getattr(cf, "cid", None) or getattr(cf, "client_proxy", None)
            if cid is not None:
                cid = cid.cid if hasattr(cid, "cid") else cid
                self._obs_history[cid].append(1)

        agg = super().aggregate_fit(server_round, results, failures)

        # virtual queue update with empirical g_i:
        rho_now = np.mean([
            dropout_probability(np.asarray(self._obs_history[cid] or [0]), self.hmm)
            for cid in self._obs_history
        ]) if self._obs_history else 0.0
        cfg_round = results[0][1].metrics if results else {}
        pq_scheme = cfg_round.get("pq_scheme", next(iter(self.cfg.pq_set)))
        n_sel = max(len(results), 1)
        action_proxy = type("A", (), {
            "selected": np.ones(n_sel, dtype=int),
            "pq_scheme": pq_scheme,
            "bandwidth": np.ones(n_sel),
        })()
        # constraints() indexes rho[selected]; rho must match selected length (not scalar).
        proxy_state = State(
            rho=np.full(n_sel, rho_now),
            snr=np.full(n_sel, 20.0),
            queue_lengths=self._Q,
            pq_service_time=self._pq_cache,
        )
        g = constraints(proxy_state, action_proxy)
        self._Q = update_virtual_queues(self._Q, g)

        self.metrics.append({
            "round": server_round,
            "n_received": len(results),
            "n_failed": len(failures),
            "rho_mean": float(rho_now),
            "Q": self._Q.tolist(),
            "pq_scheme": pq_scheme,
            "ts": time.time(),
        })
        return agg

## 11. PQ profile, client factory, centralized evaluation

In [ ]:
# Mean PQ service times (liboqs if available, else deterministic stub)
pq_means: dict[str, float] = {algo: measure(algo, n=20).mean for algo in PQ_SET}
print("PQ mean service times (s):", json.dumps(pq_means, indent=2))


def pq_sleep_sample(algo: str) -> float:
    return float(np.random.exponential(pq_means.get(algo, 0.01)))


def client_fn(cid: str) -> fl.client.Client:
    i = int(cid)
    return QRQClient(
        client_id=cid,
        model_factory=lambda: small_cnn(NUM_CLASSES, IN_CHANNELS),
        train_loader=train_loaders[i],
        val_loader=val_loader,
        device=CLIENT_DEVICE,
        local_epochs=LOCAL_EPOCHS,
        pq_overhead_sampler=pq_sleep_sample,
    ).to_client()


def make_evaluate_fn():
    def evaluate_fn(server_round: int, parameters, config: dict):
        if isinstance(parameters, Parameters):
            arrays = parameters_to_ndarrays(parameters)
        else:
            arrays = parameters  # some flwr versions pass ndarrays directly
        model = small_cnn(NUM_CLASSES, IN_CHANNELS).to(DEVICE)
        ckpt = {
            k: torch.tensor(arr, device=DEVICE)
            for k, arr in zip(model.state_dict().keys(), arrays)
        }
        model.load_state_dict(ckpt, strict=True)
        model.eval()
        loss_fn = nn.CrossEntropyLoss(reduction="sum")
        correct = total = 0
        loss_sum = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss_sum += loss_fn(logits, yb).item()
                correct += (logits.argmax(1) == yb).sum().item()
                total += len(yb)
        acc = correct / max(total, 1)
        loss = loss_sum / max(total, 1)
        return float(loss), {"accuracy": float(acc)}

    return evaluate_fn


evaluate_fn = make_evaluate_fn()


## 12. Run simulations

Two paths are available:

* **Quick path** (this section): one FedAvg baseline + one QRQ-FL run. This is what the verified Section IV table is built from.
* **Publication baselines** (Section 15): runs the full grid (FedAvg, FedAvg+Kyber, FedProx, QRQ-FL, and three QRQ-FL ablations) and writes consolidated artefacts under `runs/baselines_<timestamp>/`. Use that section when you regenerate the paper tables.


In [ ]:
RUN_DIR = pathlib.Path("runs") / time.strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts -> {RUN_DIR.resolve()}")

hist_fedavg = None
if RUN_FEDAVG_BASELINE:
    print("\n=== FedAvg baseline (random client sampling) ===")
    strat_fed = fl.server.strategy.FedAvg(
        fraction_fit=float(N_SELECT) / float(N_CLIENTS),
        fraction_evaluate=0.0,
        min_fit_clients=N_SELECT,
        min_evaluate_clients=1,
        min_available_clients=N_CLIENTS,
        evaluate_fn=evaluate_fn,
    )

    def client_fn_plain(cid: str) -> fl.client.Client:
        i = int(cid)
        return QRQClient(
            client_id=cid,
            model_factory=lambda: small_cnn(NUM_CLASSES, IN_CHANNELS),
            train_loader=train_loaders[i],
            val_loader=val_loader,
            device=CLIENT_DEVICE,
            local_epochs=LOCAL_EPOCHS,
            pq_overhead_sampler=lambda _a: 0.0,
        ).to_client()

    hist_fedavg = fl.simulation.start_simulation(
        client_fn=client_fn_plain,
        num_clients=N_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=N_ROUNDS),
        strategy=strat_fed,
        client_resources={"num_cpus": 1, "num_gpus": 0.0},
    )
    print(hist_fedavg)

print("\n=== QRQ-FL (HMM + DPP + PQ library) ===")
strategy_qrq = QRQFLStrategy(
    fraction_fit=0.0,
    fraction_evaluate=0.0,
    min_fit_clients=N_SELECT,
    min_evaluate_clients=1,
    min_available_clients=N_CLIENTS,
    evaluate_fn=evaluate_fn,
    qrq=QRQConfig(n_select=N_SELECT, V=DPP_V, pq_set=tuple(PQ_SET), deadline_s=QRQ_DEADLINE_S),
    hmm=HMMParams.reasonable_default(),
)

hist_qrq = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=N_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=N_ROUNDS),
    strategy=strategy_qrq,
    client_resources={"num_cpus": 1, "num_gpus": 0.0},
)
print(hist_qrq)

# Save server-side QRQ metrics (same shape as main.py CSV)
with (RUN_DIR / "metrics_qrq.csv").open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["round", "n_received", "n_failed", "rho_mean", "Q0", "Q1", "Q2", "pq_scheme", "ts"])
    for m in strategy_qrq.metrics:
        w.writerow(
            [
                m["round"],
                m["n_received"],
                m["n_failed"],
                m["rho_mean"],
                *m["Q"],
                m["pq_scheme"],
                m["ts"],
            ]
        )
print(f"Wrote {RUN_DIR / 'metrics_qrq.csv'}")


## 13. Accuracy curves and QRQ metrics

In [ ]:
def series_accuracy(hist) -> tuple[list[int], list[float]]:
    if hist is None:
        return [], []
    mc = getattr(hist, "metrics_centralized", None) or {}
    acc = mc.get("accuracy", [])
    rounds = [int(r) for r, _ in acc]
    vals = [float(v) for _, v in acc]
    return rounds, vals


r2, a2 = series_accuracy(hist_qrq)
plt.figure(figsize=(7, 4))
plt.plot(r2, a2, marker="s", markersize=4, label="QRQ-FL")
if hist_fedavg is not None:
    r1, a1 = series_accuracy(hist_fedavg)
    plt.plot(r1, a1, marker="o", markersize=4, label="FedAvg")
plt.xlabel("Round")
plt.ylabel("Test accuracy")
plt.title(f"{DATASET.upper()} | N={N_CLIENTS}, K={N_SELECT}, α={DIRICHLET_ALPHA}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
FIG_DIR = pathlib.Path("figures")
FIG_DIR.mkdir(exist_ok=True)
plt.savefig(FIG_DIR / "acc_centralized.pdf")
plt.savefig(IMG_PNG := FIG_DIR / "acc_centralized.png", dpi=150)
plt.show()

# PQ scheme picked per round
if strategy_qrq.metrics:
    rounds = [m["round"] for m in strategy_qrq.metrics]
    schemes = [m["pq_scheme"] for m in strategy_qrq.metrics]
    rho = [m["rho_mean"] for m in strategy_qrq.metrics]
    dfm = pd.DataFrame({"round": rounds, "pq_scheme": schemes, "rho_mean": rho})
    print(dfm.tail(10).to_string())


## 14. `results.json` and downloads

In [ ]:
def acc_dict(hist):
    _, vals = series_accuracy(hist)
    return vals


summary = {
    "config": {
        "dataset": DATASET,
        "n_clients": N_CLIENTS,
        "n_select": N_SELECT,
        "n_rounds": N_ROUNDS,
        "batch_size": BATCH_SIZE,
        "alpha": DIRICHLET_ALPHA,
        "local_epochs": LOCAL_EPOCHS,
        "lr": LR,
        "dpp_V": DPP_V,
        "pq_set": list(PQ_SET),
        "device": DEVICE,
        "seed": SEED,
    },
    "fedavg_test_acc_per_round": acc_dict(hist_fedavg) if hist_fedavg else [],
    "qrqfl_test_acc_per_round": acc_dict(hist_qrq),
    "qrqfl_server_metrics": strategy_qrq.metrics,
    "pq_mean_seconds": pq_means,
}

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Wrote results.json")

if COLAB:
    import shutil

    shutil.make_archive("figures", "zip", str(FIG_DIR))
    shutil.make_archive("run_artifacts", "zip", str(RUN_DIR))
    from google.colab import files

    files.download("figures.zip")
    files.download("run_artifacts.zip")
    files.download("results.json")
else:
    print("Local run: see figures/, results.json, and runs/")


## 15. Publication baseline grid (optional)

Run the full method grid (FedAvg, FedAvg+Kyber, FedProx, QRQ-FL, and three
QRQ-FL ablations) under the same protocol as above. Produces a consolidated
``summary.json`` and ``accuracy_curves.csv`` that can be fed into
``code/impl/build_paper_tables.py`` to regenerate the LaTeX tables and
figures included from ``main.tex``.

Skip this cell if you only need the quick FedAvg-vs-QRQ-FL trace from
Section 12.


In [ ]:
RUN_BASELINE_GRID = False  # set True to actually launch all 7 methods

if RUN_BASELINE_GRID:
    import importlib, sys
    # Bring run_baselines.py into the kernel even though Colab clones the repo flat.
    sys.path.insert(0, str(pathlib.Path('.').resolve()))
    try:
        from code.impl import run_baselines
    except ImportError:
        import run_baselines  # type: ignore
    importlib.reload(run_baselines)

    out = pathlib.Path('runs') / time.strftime('baselines_%Y%m%d_%H%M%S')
    run_baselines.main([
        '--dataset', DATASET,
        '--clients', str(N_CLIENTS),
        '--select',  str(N_SELECT),
        '--rounds',  str(N_ROUNDS),
        '--alpha',   str(DIRICHLET_ALPHA),
        '--batch',   str(BATCH_SIZE),
        '--seed',    str(SEED),
        '--out',     str(out),
    ])
    print(f'Done. Pass {out} to build_paper_tables.py to refresh paper artefacts.')
else:
    print('Set RUN_BASELINE_GRID=True to run all 7 publication baselines.')
